# 19번. 마이클 포터 5 Forces 산업위험 스코어링

## 개요
마이클 포터의 **5 Forces 프레임워크** 를 계량화하여 도소매업의 산업 구조적 위험을 스코어링합니다.

| Force | 지표 | 해석 |
|-------|------|------|
| **F1** 경쟁 강도 | 영업이익률 (상대값) | 낮을수록 가격경쟁 심화 → 위험↑ |
| **F2** 신규 진입 위협 | 유형자산/매출액 (상대값) | 낮을수록 진입장벽 낮음 → 위험↑ |
| **F3** 대체재 위협 | 산업성장률 − GDP성장률 (상대값) | 음수일수록 수요 이탈 → 위험↑ |
| **F4** 공급자 교섭력 | DPO (상대값) | 낮을수록 빠른 지급 = 교섭력 약함 → 위험↑ |
| **F5** 구매자 교섭력 | DSO (상대값) | 높을수록 느린 회수 = 교섭력 강함 → 위험↑ |

## 정규화 원칙
$$F_t = \text{지표}_{G,t} - \overline{\text{지표}}_{\text{12개 대분류}, t}$$

각 연도별로 도소매업 값에서 **12개 대분류 단순평균** 을 차감합니다.
부호 반전 후 양수 = 해당 Force 위험 높음.


In [ ]:
import sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')

import re
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


## 1. 파라미터 및 경로 설정

- `TARGET_IND`: 분석 대상 산업 코드 (다른 산업 분석 시 이 두 줄만 변경)
- `SCORE_YEARS`: 결과 출력 연도 (2022~2024)
- `YEAR_COLS`: 데이터 로드 범위 (2012~2024, 성장률 계산을 위해 전체 기간 필요)


In [ ]:
BASE = Path().resolve() / '19번 마이클 포터'

IS_CSV  = BASE / '손익계산서(제11차 한국표준산업분류, 2009~)_11184246.csv'
BS_CSV  = BASE / '재무상태표(제11차 한국표준산업분류, 2009~)_11170003.csv'
GDP_CSV = BASE / '국내총생산과 지출(명목, 연간)_10170318.csv'
OUT_CSV    = BASE / '산업지표_통합.csv'
OUT_PQ     = BASE / '산업지표_통합.parquet'

TARGET_IND   = 'G 도매 및 소매업'   # ← 다른 산업 분석 시 이 줄만 변경
TARGET_LABEL = '도소매업'           # ← 출력 파일명에 사용
OUT_F_CSV    = BASE / f'마이클 포터 5F_{TARGET_LABEL}.csv'
OUT_F_PQ     = BASE / f'마이클 포터 5F_{TARGET_LABEL}.parquet'

SCORE_YEARS = [2022, 2023, 2024]
YEAR_COLS   = [str(y) for y in range(2012, 2025)]
TOP_PATTERN = re.compile(r'^[A-Z]\s')

## 2. 헬퍼 함수

### process()
CSV → 대분류 코드만 필터 → wide(연도 컬럼) to long → 계정항목 pivot

### load_gdp()
GDP CSV에서 **전기대비증감률(%) 행**만 추출하여 연도별 성장률 Series로 반환


In [ ]:
def process(path):
    """손익/재무 CSV → 대분류 × 연도 × 계정 형태로 변환"""
    df = pd.read_csv(path, encoding='utf-8-sig')
    df['업종코드'] = df['업종코드'].str.strip()
    mask = df['업종코드'].str.match(TOP_PATTERN) | (df['업종코드'] == '전산업')
    df = df[mask].copy()
    yr_cols = [c for c in YEAR_COLS if c in df.columns]
    df_long = df.melt(id_vars=['업종코드','계정항목'], value_vars=yr_cols,
                      var_name='연도', value_name='값')
    df_long['연도'] = df_long['연도'].astype(int)
    df_long['값'] = (
        df_long['값'].astype(str).str.replace(',','',regex=False)
        .replace('-', pd.NA).pipe(pd.to_numeric, errors='coerce')
    )
    df_pivot = df_long.pivot_table(
        index=['업종코드','연도'], columns='계정항목', values='값', aggfunc='first'
    ).reset_index()
    df_pivot.columns.name = None
    return df_pivot


def load_gdp(path):
    """GDP CSV에서 전기대비증감률 행 추출 → 연도별 성장률 DataFrame"""
    df = pd.read_csv(path, encoding='utf-8-sig')
    row = df[df['변환'] == '전기대비증감률'].iloc[0]
    yr_cols = [c for c in df.columns if str(c).isdigit()]
    return pd.DataFrame({
        '연도': [int(c) for c in yr_cols],
        'GDP성장률': [float(str(row[c]).replace(',','')) for c in yr_cols],
    })


## 3. 데이터 로드 및 병합

3개 소스를 연도 기준으로 병합합니다:
1. **손익계산서**: 매출액, 매출원가, 영업손익 (단위: 백만원)
2. **재무상태표**: 매출채권, 유형자산, 매입채무 (단위: 백만원)
3. **GDP 성장률**: 명목 GDP 전기대비증감률 (단위: %)

> 손익/재무 두 파일 모두 **백만원 단위**, 단위 불일치 없음


In [ ]:
is_df  = process(IS_CSV)
bs_df  = process(BS_CSV)
gdp_df = load_gdp(GDP_CSV)

print(f'손익계산서: {is_df.shape}  계정: {[c for c in is_df.columns if c not in ["업종코드","연도"]]}')
print(f'재무상태표: {bs_df.shape}  계정: {[c for c in bs_df.columns if c not in ["업종코드","연도"]]}')
print(f'GDP 성장률: {len(gdp_df)}개 연도')

merged = is_df.merge(bs_df, on=['업종코드','연도'], how='outer')
merged = merged.merge(gdp_df, on='연도', how='left')
merged = merged.sort_values(['업종코드','연도']).reset_index(drop=True)

col_order = ['업종코드','연도','매출액','매출원가','영업손익','매출채권','유형자산','매입채무','GDP성장률']
merged = merged[[c for c in col_order if c in merged.columns]]

print(f'\n병합 결과: {merged.shape}  연도: {merged["연도"].min()}~{merged["연도"].max()}')
print(f'업종 수: {merged["업종코드"].nunique()}개  (12개 대분류 + 전산업)')

merged.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
merged.to_parquet(OUT_PQ, index=False)
print(f'\n원본 저장: {OUT_CSV}')


## 4. F1~F5 스코어 계산

### 계산 순서
1. 전산업 행 제외, 12개 대분류만 사용
2. 각 산업별 지표 계산 (영업이익률, 유형자산비율, DPO, DSO, 매출성장률)
3. F3_raw = 산업 매출성장률 − GDP성장률
4. 연도별 12개 산업 단순평균 계산 → 벤치마크
5. adj = 대상 산업 값 − 벤치마크
6. 부호 반전 적용 → F1~F5 (양수 = 위험 높음)

### 부호 반전 규칙
| Force | 방향 | 이유 |
|-------|------|------|
| F1 (영업이익률) | 반전 | 낮을수록 위험 |
| F2 (유형자산비율) | 반전 | 낮을수록 진입장벽 낮음 |
| F3 (초과성장률) | 반전 | 음수일수록 대체재 침투 |
| F4 (DPO) | 반전 | 낮을수록 공급자 교섭력 약함 |
| F5 (DSO) | 유지 | 높을수록 구매자 교섭력 강함 |


In [ ]:
# 전산업 제외, 12개 대분류만
ind_only = merged[merged['업종코드'] != '전산업'].copy().sort_values(['업종코드','연도'])

# 지표 계산
ind_only['영업이익률']  = ind_only['영업손익'] / ind_only['매출액']
ind_only['유형자산비율'] = ind_only['유형자산'] / ind_only['매출액']
ind_only['DPO']         = ind_only['매입채무'] / (ind_only['매출원가'] / 365)
ind_only['DSO']         = ind_only['매출채권'] / (ind_only['매출액']   / 365)
ind_only['매출성장률']  = ind_only.groupby('업종코드')['매출액'].pct_change() * 100
ind_only['F3_raw']      = ind_only['매출성장률'] - ind_only['GDP성장률']

# 연도별 단순 평균 벤치마크 (12개 업종 동일 가중치)
METRICS = ['영업이익률','유형자산비율','DPO','DSO','F3_raw']
bench = (
    ind_only.groupby('연도')[METRICS].mean().reset_index()
    .rename(columns={c: c+'_bench' for c in METRICS})
)

# 대상 산업 adj 계산
g_ind = ind_only[ind_only['업종코드'] == TARGET_IND].copy()
g_cmp = g_ind.merge(bench, on='연도', how='left')
for col in METRICS:
    g_cmp[col+'_adj'] = g_cmp[col] - g_cmp[col+'_bench']

# F1~F5 부호 적용
g_cmp['F1'] = -g_cmp['영업이익률_adj']   # 낮을수록 위험 → 반전
g_cmp['F2'] = -g_cmp['유형자산비율_adj']  # 낮을수록 위험 → 반전
g_cmp['F3'] = -g_cmp['F3_raw_adj']        # 음수일수록 위험 → 반전
g_cmp['F4'] = -g_cmp['DPO_adj']           # 낮을수록 위험 → 반전
g_cmp['F5'] =  g_cmp['DSO_adj']           # 높을수록 위험 → 유지

result = g_cmp[g_cmp['연도'].isin(SCORE_YEARS)][['연도','F1','F2','F3','F4','F5']].copy()
result = result.sort_values('연도').reset_index(drop=True)
result.insert(0, '산업명', TARGET_LABEL)

print(f'\n{"":>6} {"F1 경쟁강도":>12} {"F2 진입장벽":>12} {"F3 대체재":>12} {"F4 공급자":>12} {"F5 구매자":>12}')
print('-' * 68)
for _, row in result.iterrows():
    print(f'{row["산업명"]:>6} {int(row["연도"]):>6} {row["F1"]:>+12.4f} {row["F2"]:>+12.4f} {row["F3"]:>+12.4f} {row["F4"]:>+12.4f} {row["F5"]:>+12.4f}')
print('\n※ 양수(+) = 위험 높음  |  음수(-) = 상대적 위험 낮음')


## 5. 저장

In [ ]:
result.to_csv(OUT_F_CSV, index=False, encoding='utf-8-sig')
result.to_parquet(OUT_F_PQ, index=False)
print(f'저장 완료: {OUT_F_CSV}')
print(result.to_string(index=False))
